In [ ]:
# 0 · which GPU are we on? (VERL wants A100-class; T4 16GB will likely OOM)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [3]:
# 1 · install the VERL backend (heavy: verl + vllm + ray). If Colab asks to
#     restart the runtime after this, do it, then run from cell 1 again.
!pip install -q -U "shadowlm[verl]"
import shadowlm as slm
print("shadowlm", slm.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 16.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 96.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 12.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.0/450.0 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.

In [4]:
%%writefile lyzr_reward.py
# VERL imports the reward by path, so it must be a named fn in an importable
# module — not a lambda or a function defined in the notebook (__main__).
def reward(prompts, completions, answer=None, **k):
    # toy reward: 1.0 if the completion mentions paris
    return [1.0 if "paris" in (c or "").lower() else 0.0 for c in completions]

Writing lyzr_reward.py


In [ ]:
# 3 · smoke run — tiny GRPO via VERL (just proves the pipe end to end)
# note: backend is set at load(); finetune() does NOT take a backend arg.
import lyzr_reward, shadowlm as slm
m = slm.load("Qwen/Qwen2.5-0.5B-Instruct", backend="verl")
data = [{"prompt": "What is the capital of France?", "answer": "Paris"}] * 32
run = m.finetune(data, method="grpo", reward_fns=[lyzr_reward.reward],
                 max_steps=5, grpo_group_size=4)
print("checkpoint:", run.checkpoint)